# 7교시. 추출 결과 검증 및 데이터 저장

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/document_ai_lecture_2026/colab/07_validation_export.ipynb)

**목표:** 필수값과 품목 합계를 확인하고 검증된 결과만 Excel로 저장합니다.

**결과물:** `receipt_result.xlsx`

- 모든 필수 실습은 Google Colab에서 진행합니다.
- API 키나 결제가 필요 없습니다.
- 식별정보를 가린 교육용 샘플 한 장만 사용합니다.
- 개인·회사 문서를 외부 API에 보내거나 공개 Streamlit 주소를 만들지 않습니다.
- 실행이 3분을 넘으면 중지하고 준비 결과를 선택합니다.
- 필요한 셀을 위에서 아래로 다시 실행하고, 계속 실패하면 강사에게 알립니다.
- 실습이 끝나면 Colab 출력과 런타임 파일을 삭제하고 런타임을 종료합니다.


In [ ]:
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Output:", OUTPUT_DIR.resolve())


In [ ]:
import importlib.util
import subprocess

if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "openpyxl==3.1.5",
        ]
    )


In [ ]:
from copy import deepcopy
from datetime import date
from openpyxl import Workbook, load_workbook

SAMPLE_RECEIPT = {'document_type': 'receipt', 'store_name': '샘플문구점', 'date': '2026-07-27', 'total_amount': 5000, 'items': [{'name': '연필', 'quantity': 2, 'unit_price': 1000, 'line_total': 2000}, {'name': '노트', 'quantity': 1, 'unit_price': 3000, 'line_total': 3000}], 'source_mode': 'prepared'}
SAMPLE_OCR_TEXT = '샘플문구점\n거래일자: 2026-07-27\n연필 2개 × 1,000원 = 2,000원\n노트 1개 × 3,000원 = 3,000원\n합계: 5,000원\n'
MISSING_STORE = deepcopy(SAMPLE_RECEIPT)
MISSING_STORE["store_name"] = None
WRONG_TOTAL = deepcopy(SAMPLE_RECEIPT)
WRONG_TOTAL["total_amount"] = 6000
BAD_DATE = deepcopy(SAMPLE_RECEIPT)
BAD_DATE["date"] = "2026/07/27"
BAD_AMOUNT = deepcopy(SAMPLE_RECEIPT)
BAD_AMOUNT["total_amount"] = "5,000원"


## 핵심 3개

1. 검증 결과는 valid·warnings·errors로 나눕니다.
2. 자료형과 업무 규칙은 다른 검사입니다.
3. 오류가 없고 사람이 확인한 결과만 Excel로 저장합니다.


In [ ]:
def is_iso_date(value):
    if not isinstance(value, str):
        return False
    try:
        date.fromisoformat(value)
        return True
    except ValueError:
        return False


def validate_receipt(data):
    errors = []

    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")

    if data.get("date") and not is_iso_date(data["date"]):
        errors.append("date는 YYYY-MM-DD 형식이어야 합니다.")

    total_amount = data.get("total_amount")
    if total_amount is not None and (
        not isinstance(total_amount, int) or total_amount < 0
    ):
        errors.append("total_amount는 0 이상의 정수여야 합니다.")

    item_sum = sum(
        item.get("line_total", 0)
        for item in data.get("items", [])
        if isinstance(item.get("line_total"), int)
    )
    if isinstance(total_amount, int) and total_amount != item_sum:
        errors.append("품목 합계와 총액이 다릅니다.")

    return {
        "valid": not errors,
        "warnings": [],
        "errors": errors,
    }


## 실습. 다섯 데이터 검증


In [ ]:
normal = validate_receipt(SAMPLE_RECEIPT)
missing = validate_receipt(MISSING_STORE)
wrong_total = validate_receipt(WRONG_TOTAL)
bad_date = validate_receipt(BAD_DATE)
bad_amount = validate_receipt(BAD_AMOUNT)

assert normal["valid"]
assert not missing["valid"]
assert not wrong_total["valid"]
assert not bad_date["valid"]
assert not bad_amount["valid"]

print("정상:", normal)
print("누락:", missing)
print("합계 불일치:", wrong_total)
print("날짜 형식:", bad_date)
print("금액 형식:", bad_amount)


In [ ]:
def safe_text(value):
    if isinstance(value, str) and value.startswith(("=", "+", "-", "@")):
        return "'" + value
    return value


def receipt_rows(data):
    rows = []
    for item in data["items"]:
        row = {
            "store_name": data["store_name"],
            "date": data["date"],
            "total_amount": data["total_amount"],
            "item_name": item["name"],
            "quantity": item["quantity"],
            "unit_price": item["unit_price"],
            "line_total": item["line_total"],
        }
        rows.append({key: safe_text(value) for key, value in row.items()})
    return rows


def save_reviewed_excel(
    data,
    validation,
    *,
    human_approved,
    output_path,
    source_text,
):
    if not validation["valid"] or not human_approved:
        return False

    raw_values = {
        "store_name": data["store_name"],
        "date": data["date"],
        "total_amount": f'{data["total_amount"]:,}원',
    }
    workbook = Workbook()
    summary = workbook.active
    summary.title = "검토_요약"
    summary.append(
        [
            "field",
            "raw_value",
            "cleaned_value",
            "final_value",
            "review_status",
        ]
    )
    for field in ("store_name", "date", "total_amount"):
        summary.append(
            [
                field,
                safe_text(raw_values[field]),
                safe_text(data[field]),
                safe_text(data[field]),
                "사람 확인 완료",
            ]
        )

    items = workbook.create_sheet("품목")
    columns = [
        "store_name", "date", "total_amount", "item_name",
        "quantity", "unit_price", "line_total",
    ]
    items.append(columns)
    for row in receipt_rows(data):
        items.append([row[column] for column in columns])

    source = workbook.create_sheet("원문")
    source.append(["source_mode", data["source_mode"]])
    source.append(["ocr_text", safe_text(source_text)])
    workbook.save(output_path)
    return True


blocked_path = OUTPUT_DIR / "blocked_result.xlsx"
assert not save_reviewed_excel(
    WRONG_TOTAL,
    wrong_total,
    human_approved=True,
    output_path=blocked_path,
    source_text=SAMPLE_OCR_TEXT,
)
assert not blocked_path.exists()

not_approved_path = OUTPUT_DIR / "not_approved.xlsx"
assert not save_reviewed_excel(
    SAMPLE_RECEIPT,
    normal,
    human_approved=False,
    output_path=not_approved_path,
    source_text=SAMPLE_OCR_TEXT,
)
assert not not_approved_path.exists()
print("미승인 차단 확인:", not not_approved_path.exists())


## 원본을 확인한 뒤 직접 승인하기

1. 원본에서 상호명·날짜·총액을 확인합니다.
2. 값이 모두 같을 때만 아래 셀의 `False`를 `True`로 바꿉니다.
3. 셀을 다시 실행해 `receipt_result.xlsx`를 만듭니다.

전체 정답을 불러와도 승인값은 `False`로 시작합니다.


In [ ]:
HUMAN_APPROVED = False
output_path = OUTPUT_DIR / "receipt_result.xlsx"

if not HUMAN_APPROVED:
    print("원본 확인 전입니다. Excel을 만들지 않습니다.")
else:
    assert save_reviewed_excel(
        SAMPLE_RECEIPT,
        normal,
        human_approved=HUMAN_APPROVED,
        output_path=output_path,
        source_text=SAMPLE_OCR_TEXT,
    )
    saved = load_workbook(output_path)
    assert saved.sheetnames == ["검토_요약", "품목", "원문"]
    assert [cell.value for cell in saved["검토_요약"][1]] == [
        "field",
        "raw_value",
        "cleaned_value",
        "final_value",
        "review_status",
    ]
    print("저장 완료:", output_path, saved.sheetnames)


In [ ]:
RUN_COLAB_DOWNLOAD = False

if RUN_COLAB_DOWNLOAD and output_path.exists():
    from google.colab import files
    files.download(str(output_path))
elif RUN_COLAB_DOWNLOAD:
    print("원본 확인과 사람 승인을 먼저 완료하세요.")
else:
    print("자동 다운로드를 건너뛰었습니다. Colab 파일 영역에서 받을 수 있습니다.")


## 준비 결과 경로

이전 앱 없이 내장 `SAMPLE_RECEIPT`를 같은 검증·Excel 함수에 전달합니다.
